# Prediksi Kelulusan Ujian Siswa (`pass_status`) — Eksplorasi & Pra-pemrosesan Data

Notebook ini menyiapkan data `student_exam_performance.csv` (100.000 baris, 44 kolom) untuk klasifikasi **Pass / Fail**.

**Alur kerja**
1. Eksplorasi awal dataset
2. Pemilihan fitur (13 fitur X + 1 target y) dan pembuangan kolom
3. Skala pengukuran & ordinal encoding
4. Pra-pemrosesan: duplikat, missing value, split, outlier, winsorizing, encoding, oversampling
5. Data akhir `X_train`, `y_train`, `X_test`, `y_test`

> **Prinsip anti-leakage:** semua statistik (batas IQR, dll.) dihitung dari **data latih saja**, lalu diterapkan ke data uji. Oversampling hanya pada data latih.

## 0. Setup

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

FILE_PATH = "data/student_exam_performance.csv"
RANDOM_STATE = 42
TARGET = "pass_status"

pd.set_option("display.width", 140)
df = pd.read_csv(FILE_PATH)

## 1. Eksplorasi Awal Dataset
Mengecek ukuran data, missing value, duplikat, dan distribusi kelas pada ketiga kolom hasil (`pass_status`, `performance_grade`, `performance_level`).

In [2]:
print(f"Ukuran dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print(f"Baris duplikat: {df.duplicated().sum()}")

# Missing value (hanya kolom yang memiliki NaN)
missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"\nTotal missing value: {missing.sum():,}")
print(pd.DataFrame({"Jumlah": missing, "Persen (%)": (missing / len(df) * 100).round(2)}))

# Distribusi ketiga kolom hasil dalam satu tabel ringkas
for col in ["pass_status", "performance_grade", "performance_level"]:
    vc = df[col].value_counts()
    print(f"\n{col}:")
    print(pd.DataFrame({"Jumlah": vc, "Persen (%)": (vc / len(df) * 100).round(2)}))

Ukuran dataset: 100,000 baris x 44 kolom
Baris duplikat: 0

Total missing value: 52,161
                       Jumlah  Persen (%)
parent_education         6526        6.53
previous_gpa             7793        7.79
attendance_percentage    9863        9.86
notes_quality            8407        8.41
sleep_quality            6994        6.99
device_availability      2965        2.96
time_management_score    9613        9.61

pass_status:
             Jumlah  Persen (%)
pass_status                    
Pass          77389       77.39
Fail          22611       22.61

performance_grade:
                   Jumlah  Persen (%)
performance_grade                    
D                   34886       34.89
F                   22611       22.61
C                   21794       21.79
B                   13190       13.19
A                    7519        7.52

performance_level:
                   Jumlah  Persen (%)
performance_level                    
Low                 45123       45.12
Medium        

## 2. Pemilihan Fitur dan Target

| Peran | Jumlah | Keterangan |
|---|---|---|
| Fitur X | 13 | riwayat akademik, kebiasaan belajar, gaya hidup/mental, kondisi ujian, latar belakang |
| Target y | 1 | `pass_status` |
| Dibuang: leakage / ID | 5 | `exam_score`, `performance_grade`, `performance_level`, `questions_correct` (bocor dari hasil ujian), `student_id` (pengenal) |
| Dibuang: tidak relevan | 25 | tidak dipakai pada model |

Rincian 44 kolom tidak dicetak ulang di sini; cukup ringkasan dan validasi lewat `assert`.

In [3]:
FITUR_PER_KELOMPOK = {
    "Riwayat akademik": ["previous_exam_score", "previous_gpa", "attendance_percentage", "assignment_completion_rate"],
    "Kebiasaan belajar": ["study_hours_per_day", "self_study_hours", "practice_tests_completed", "exam_preparation_days"],
    "Gaya hidup / mental": ["sleep_hours", "stress_level", "exam_anxiety_level"],
    "Kondisi ujian": ["exam_difficulty"],
    "Latar belakang": ["family_income"],
}
fitur = [f for cols in FITUR_PER_KELOMPOK.values() for f in cols]

kolom_leakage_id = ["exam_score", "performance_grade", "performance_level", "questions_correct", "student_id"]

kolom_tidak_relevan = [
    "age", "gender", "education_level", "school_type", "parent_education", "urban_rural",
    "class_participation", "study_consistency", "study_environment", "study_method",
    "revision_frequency", "notes_quality", "private_tuition", "time_management_score",
    "sleep_quality", "daily_screen_time", "physical_activity_hours", "break_frequency",
    "motivation_level", "internet_access", "device_availability",
    "educational_app_usage", "online_learning_hours", "online_course_hours",
    "questions_attempted",
]

# Validasi: tiap kolom terdaftar tepat satu kali dan cocok dengan CSV
semua = fitur + [TARGET] + kolom_leakage_id + kolom_tidak_relevan
assert len(semua) == len(set(semua)), "Ada kolom terdaftar ganda!"
assert set(semua) == set(df.columns), "Kolom tidak sesuai dengan dataset!"

print(pd.DataFrame({
    "Peran": ["Fitur (X)", "Target (y)", "Dibuang (leakage & ID)", "Dibuang (tidak relevan)", "Total"],
    "Jumlah kolom": [len(fitur), 1, len(kolom_leakage_id), len(kolom_tidak_relevan), len(semua)],
}).to_string(index=False))

print("\nFitur terpilih per kelompok:")
for kel, cols in FITUR_PER_KELOMPOK.items():
    print(f"- {kel}: {', '.join(cols)}")

                  Peran  Jumlah kolom
              Fitur (X)            13
             Target (y)             1
 Dibuang (leakage & ID)             5
Dibuang (tidak relevan)            25
                  Total            44

Fitur terpilih per kelompok:
- Riwayat akademik: previous_exam_score, previous_gpa, attendance_percentage, assignment_completion_rate
- Kebiasaan belajar: study_hours_per_day, self_study_hours, practice_tests_completed, exam_preparation_days
- Gaya hidup / mental: sleep_hours, stress_level, exam_anxiety_level
- Kondisi ujian: exam_difficulty
- Latar belakang: family_income


### Subset data: 13 fitur + 1 target

In [4]:
data = df[fitur + [TARGET]].copy()
assert data.shape[1] == 14
data.to_csv("data/data_dipilih.csv", index=False)  # opsional
print("Subset data:", data.shape)

Subset data: (100000, 14)


## 3. Skala Pengukuran
Pengelompokan 14 kolom berdasarkan skala: **Nominal** (kategori tanpa urutan), **Ordinal** (kategori berurutan), **Interval** (numerik tanpa nol mutlak), **Rasio** (numerik dengan nol mutlak).

In [5]:
skala_ukur = {
    TARGET: "Nominal",
    "exam_difficulty": "Ordinal", "family_income": "Ordinal",
    "previous_exam_score": "Interval", "previous_gpa": "Interval",
    "stress_level": "Interval", "exam_anxiety_level": "Interval",
    "attendance_percentage": "Rasio", "assignment_completion_rate": "Rasio",
    "study_hours_per_day": "Rasio", "self_study_hours": "Rasio",
    "practice_tests_completed": "Rasio", "exam_preparation_days": "Rasio",
    "sleep_hours": "Rasio",
}
assert set(skala_ukur) == set(data.columns), "Kolom skala_ukur tidak sesuai data!"

kolom_num = [c for c, s in skala_ukur.items() if s in ("Interval", "Rasio")]

# Tabel gabungan: skala + nilai min/max (menggantikan 3 tabel terpisah)
tabel_skala = pd.DataFrame({
    "Kolom": list(skala_ukur),
    "Skala": list(skala_ukur.values()),
    "Min": [data[c].min() if c in kolom_num else None for c in skala_ukur],
    "Max": [data[c].max() if c in kolom_num else None for c in skala_ukur],
})
print(tabel_skala.to_string(index=False))
print("\nRekap jumlah kolom per skala:")
print(tabel_skala["Skala"].value_counts().to_string())

                     Kolom    Skala   Min   Max
               pass_status  Nominal   NaN   NaN
           exam_difficulty  Ordinal   NaN   NaN
             family_income  Ordinal   NaN   NaN
       previous_exam_score Interval 30.00 100.0
              previous_gpa Interval  1.00   4.0
              stress_level Interval  5.00  10.0
        exam_anxiety_level Interval  1.00  10.0
     attendance_percentage    Rasio 46.34 100.0
assignment_completion_rate    Rasio 20.00 100.0
       study_hours_per_day    Rasio  0.50  12.0
          self_study_hours    Rasio  0.25  10.0
  practice_tests_completed    Rasio  0.00  20.0
     exam_preparation_days    Rasio  1.00  30.0
               sleep_hours    Rasio  3.00  11.0

Rekap jumlah kolom per skala:
Skala
Rasio       7
Interval    4
Ordinal     2
Nominal     1


## 4. Ordinal Encoding
Pemetaan didefinisikan **sekali** di sini dan dipakai ulang pada bagian pra-pemrosesan.

| Kolom | Pemetaan |
|---|---|
| `exam_difficulty` | Easy=1, Medium=2, Hard=3 |
| `family_income` | Low=1, Lower-Middle=2, Middle=3, Upper-Middle=4, High=5 |

In [6]:
mapping_exam_difficulty = {"Easy": 1, "Medium": 2, "Hard": 3}
mapping_family_income = {"Low": 1, "Lower-Middle": 2, "Middle": 3, "Upper-Middle": 4, "High": 5}

def encode_ordinal(frame):
    """Kembalikan salinan frame dengan dua kolom ordinal berupa angka."""
    out = frame.copy()
    out["exam_difficulty"] = out["exam_difficulty"].map(mapping_exam_difficulty)
    out["family_income"] = out["family_income"].map(mapping_family_income)
    assert out[["exam_difficulty", "family_income"]].isna().sum().sum() == 0, "NaN muncul setelah encoding!"
    return out

# Demonstrasi pada satu siswa (indeks 56663): sebelum vs sesudah
data_enc = encode_ordinal(data)
idx = 56663
perbandingan = pd.DataFrame({
    "Skala": pd.Series(skala_ukur),
    "Sebelum": data.loc[idx],
    "Sesudah": data_enc.loc[idx],
})
print(f"Perbandingan sebelum & sesudah encoding (siswa indeks {idx}):")
print(perbandingan.loc[["exam_difficulty", "family_income"]])

Perbandingan sebelum & sesudah encoding (siswa indeks 56663):
                   Skala Sebelum Sesudah
exam_difficulty  Ordinal    Easy       1
family_income    Ordinal    High       5


## 5. Pra-pemrosesan Data

### 5.1 Duplikat dan tipe data
`data_prep` adalah data mentah (belum di-encode) yang akan diproses mengikuti urutan yang benar: **hapus missing → split → outlier → encoding → oversampling**.

In [7]:
data_prep = df[fitur + [TARGET]].copy()
print(f"Baris duplikat: {data_prep.duplicated().sum()}")
print("\nKategori unik:")
for c in ["exam_difficulty", "family_income", TARGET]:
    print(f"- {c}: {data_prep[c].unique().tolist()}")
print("\nTipe data:")
print(data_prep.dtypes.astype(str).to_string())

Baris duplikat: 0

Kategori unik:
- exam_difficulty: ['Medium', 'Easy', 'Hard']
- family_income: ['Middle', 'High', 'Lower-Middle', 'Upper-Middle', 'Low']
- pass_status: ['Pass', 'Fail']

Tipe data:
previous_exam_score           float64
previous_gpa                  float64
attendance_percentage         float64
assignment_completion_rate    float64
study_hours_per_day           float64
self_study_hours              float64
practice_tests_completed        int64
exam_preparation_days           int64
sleep_hours                   float64
stress_level                    int64
exam_anxiety_level            float64
exam_difficulty                object
family_income                  object
pass_status                    object


### 5.2 Menghapus baris dengan nilai kosong (`dropna`)
Hanya `previous_gpa` dan `attendance_percentage` yang memiliki missing value pada 13 fitur terpilih. Proporsi kelas target dibandingkan sebelum dan sesudah untuk memastikan tidak ada bias.

In [8]:
sebelum = len(data_prep)
dist_sebelum = data_prep[TARGET].value_counts(normalize=True).mul(100).round(2)

data_clean = data_prep.dropna().copy()   # indeks tidak di-reset

dist_sesudah = data_clean[TARGET].value_counts(normalize=True).mul(100).round(2)
print(f"Baris: {sebelum:,} -> {len(data_clean):,}  (dibuang {sebelum - len(data_clean):,} = {(sebelum - len(data_clean)) / sebelum * 100:.2f}%)")
print("\nProporsi target (%):")
print(pd.DataFrame({"Sebelum": dist_sebelum, "Sesudah": dist_sesudah}))

Baris: 100,000 -> 83,125  (dibuang 16,875 = 16.88%)

Proporsi target (%):
             Sebelum  Sesudah
pass_status                  
Pass           77.39    77.32
Fail           22.61    22.68


### 5.3 Split data latih dan uji (80:20, stratified)

In [9]:
train_df, test_df = train_test_split(
    data_clean, test_size=0.20, random_state=RANDOM_STATE, stratify=data_clean[TARGET]
)
train_df, test_df = train_df.copy(), test_df.copy()

print(f"Train: {train_df.shape[0]:,} baris | Test: {test_df.shape[0]:,} baris")
print("\nProporsi target (%):")
print(pd.DataFrame({
    "Train": train_df[TARGET].value_counts(normalize=True).mul(100).round(2),
    "Test": test_df[TARGET].value_counts(normalize=True).mul(100).round(2),
}))

Train: 66,500 baris | Test: 16,625 baris

Proporsi target (%):
             Train   Test
pass_status              
Pass         77.32  77.32
Fail         22.68  22.68


### 5.4 Analisis outlier (IQR 1,5x) dan skewness
Batas dihitung dari **data latih** dan disimpan di `iqr_bounds` agar bisa dipakai ke data uji.

Cara baca: `% < BB` = persentase nilai di bawah batas bawah, `% > BA` = di atas batas atas.

In [10]:
kolom_numerik = [c for c in fitur if c not in ("exam_difficulty", "family_income")]

iqr_bounds, baris = {}, []
for col in kolom_numerik:
    q1, q3 = train_df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    bb, ba = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    iqr_bounds[col] = (bb, ba)
    n_bb, n_ba = (train_df[col] < bb).sum(), (train_df[col] > ba).sum()
    baris.append({
        "Kolom": col, "Min": train_df[col].min(), "Max": train_df[col].max(),
        "BB": bb, "BA": ba,
        "% < BB": n_bb / len(train_df) * 100, "% > BA": n_ba / len(train_df) * 100,
        "Skew": train_df[col].skew(),
    })
print(pd.DataFrame(baris).round(2).to_string(index=False))

                     Kolom   Min   Max     BB     BA  % < BB  % > BA  Skew
       previous_exam_score 30.00 100.0  38.62 100.50    0.32    0.00 -0.01
              previous_gpa  1.00   4.0   1.48   4.08    0.32    0.00 -0.02
     attendance_percentage 46.34 100.0  62.78 106.58    0.38    0.00 -0.17
assignment_completion_rate 20.00 100.0  36.86 102.54    0.31    0.00  0.02
       study_hours_per_day  0.50  12.0  -1.57   7.20    0.00    2.62  1.13
          self_study_hours  0.25  10.0  -1.17   5.07    0.00    2.95  1.28
  practice_tests_completed  0.00  20.0  -0.50  11.50    0.00    1.77  0.53
     exam_preparation_days  1.00  30.0 -14.50  45.50    0.00    0.00  0.00
               sleep_hours  3.00  11.0   3.74  10.26    0.35    0.32 -0.00
              stress_level  5.00  10.0   7.50  11.50   10.09    0.00 -1.58
        exam_anxiety_level  1.00  10.0   5.58  12.66    2.98    0.00 -1.43


**Keputusan:** hanya 3 kolom yang di-*winsorize* (batas atas), yaitu yang memiliki ekor kanan nyata dan skewness positif:
`study_hours_per_day`, `self_study_hours`, `practice_tests_completed`.

Kolom lain dibiarkan: outlier-nya sangat sedikit (<0,4%) dan berupa nilai valid, atau (seperti `stress_level`, `exam_anxiety_level`) hanya menumpuk di nilai maksimum skala sehingga bukan outlier sebenarnya.

### 5.5 Winsorizing (clipping) pada 3 kolom

In [11]:
kolom_capping = ["study_hours_per_day", "self_study_hours", "practice_tests_completed"]
ringkasan = []
for col in kolom_capping:
    bb, ba = iqr_bounds[col]
    ringkasan.append({
        "Kolom": col, "BB": round(bb, 2), "BA": round(ba, 2),
        "Berubah (Train)": int(((train_df[col] < bb) | (train_df[col] > ba)).sum()),
        "Berubah (Test)": int(((test_df[col] < bb) | (test_df[col] > ba)).sum()),
    })
    train_df[col] = train_df[col].clip(lower=bb, upper=ba)
    test_df[col] = test_df[col].clip(lower=bb, upper=ba)
print(pd.DataFrame(ringkasan).to_string(index=False))

                   Kolom    BB    BA  Berubah (Train)  Berubah (Test)
     study_hours_per_day -1.57  7.20             1742             406
        self_study_hours -1.17  5.07             1963             499
practice_tests_completed -0.50 11.50             1175             276


### 5.6 Ordinal encoding pada data latih dan uji
Memakai fungsi `encode_ordinal` dari bagian 4.

In [12]:
train_df = encode_ordinal(train_df)
test_df = encode_ordinal(test_df)
print("Encoding selesai tanpa NaN pada train maupun test.")

Encoding selesai tanpa NaN pada train maupun test.


### 5.7 Oversampling kelas minoritas (hanya data latih)
Kelas `Fail` (minoritas, ±22,7%) digandakan secara acak dengan pengembalian hingga jumlahnya sama dengan kelas `Pass`. **Data uji tidak disentuh** agar evaluasi tetap mencerminkan distribusi asli.

In [13]:
mayoritas = train_df[train_df[TARGET] == "Pass"]
minoritas = train_df[train_df[TARGET] == "Fail"]

minoritas_up = resample(minoritas, replace=True, n_samples=len(mayoritas), random_state=RANDOM_STATE)
train_resampled = pd.concat([mayoritas, minoritas_up]).sample(frac=1, random_state=RANDOM_STATE)

print(pd.DataFrame({
    "Sebelum": train_df[TARGET].value_counts(),
    "Sesudah": train_resampled[TARGET].value_counts(),
}))
print(f"\nData uji tetap orisinal: {len(test_df):,} baris")

             Sebelum  Sesudah
pass_status                  
Pass           51418    51418
Fail           15082    51418

Data uji tetap orisinal: 16,625 baris


## 6. Data Akhir
Pemisahan fitur (X) dan target (y) yang siap dipakai untuk pelatihan model.

In [14]:
X_train, y_train = train_resampled.drop(columns=[TARGET]), train_resampled[TARGET]
X_test, y_test = test_df.drop(columns=[TARGET]), test_df[TARGET]

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test : {X_test.shape} | y_test : {y_test.shape}")
train_resampled.head(3)

X_train: (102836, 13) | y_train: (102836,)
X_test : (16625, 13) | y_test : (16625,)


,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,exam_preparation_days,sleep_hours,stress_level,exam_anxiety_level,exam_difficulty,family_income,pass_status
65155,76.54,3.21,84.21,74.61,1.80,1.22,8.0,13,6.13,10,10.00,2,1,Pass
71522,67.70,2.86,74.99,60.38,0.69,0.57,8.0,28,4.51,10,10.00,2,3,Fail
84297,55.58,2.28,91.08,65.58,2.95,1.80,7.0,11,6.74,9,9.38,2,3,Fail


## Catatan
- `practice_tests_completed` menjadi bertipe `float` setelah di-*clip* (batas atas 11,5). Jika model memerlukan bilangan bulat, lakukan pembulatan.
- Oversampling dengan `resample` hanya menduplikasi baris `Fail`; bisa dicoba pembobotan kelas (`class_weight`) atau SMOTE sebagai pembanding.
- Skala fitur (standardisasi) belum dilakukan; tambahkan hanya jika model yang dipakai sensitif terhadap skala (mis. KNN, SVM, regresi logistik), dan hitung dari data latih saja.